## 1. 单图紫边伪影渲染

In [1]:
import cv2
import numpy as np
import random

def add_sparse_purple_fringe(img, intensity=0.2, max_width=1, sparse_ratio=0.1):
    """
    添加稀疏的紫边伪影，适合数据增强使用。
    :param img: 输入RGB图像，np.uint8
    :param intensity: 紫边的颜色强度（建议0.1~0.3）
    :param max_width: 紫边最大宽度（建议1~2）
    :param sparse_ratio: 稀疏比例（0~1），表示有多少比例的边缘点被选中
    """
    img = img.astype(np.float32) / 255.0
    gray = cv2.cvtColor((img * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, 100, 200)

    # 稀疏采样边缘
    mask_indices = np.argwhere(edges > 0)
    selected_indices = mask_indices[np.random.choice(
        len(mask_indices), size=int(len(mask_indices) * sparse_ratio), replace=False
    )]

    mask = np.zeros_like(gray, dtype=np.float32)
    for y, x in selected_indices:
        mask[max(0, y - max_width):y + max_width + 1, max(0, x - max_width):x + max_width + 1] = 1.0

    mask = cv2.GaussianBlur(mask, (3, 3), 0)

    # 紫色图层（偏红蓝）
    purple = np.zeros_like(img)
    purple[..., 0] = 0.6  # R
    purple[..., 1] = 0.0  # G
    purple[..., 2] = 0.8  # B

    # 融合
    mask = mask[..., None]
    result = img * (1 - intensity * mask) + purple * (intensity * mask)

    return np.clip(result * 255, 0, 255).astype(np.uint8)

In [2]:
# 读取原图像
img = cv2.imread('1.jpg')
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# 添加稀疏紫边
fringe_img = add_sparse_purple_fringe(img_rgb, intensity=0.2, max_width=1, sparse_ratio=0.2)

# 保存或展示
cv2.imwrite('edge.jpg', cv2.cvtColor(fringe_img, cv2.COLOR_RGB2BGR))

True

## 2. 单图紫晕渲染

In [ ]:
import cv2
import numpy as np


def add_subtle_purple_fringe(
    img,                          # BGR uint8
    highlight_thresh=240,         # 亮度阈值：亮区才可能紫边
    grad_thresh=30,               # Sobel 梯度阈：高反差细边
    edge_width=3,                 # 像素，决定紫边宽度
    strength=0.6,                 # 0~1
    radial_gamma=2.2              # γ>1，四角更浓
):
    h, w = img.shape[:2]
    f_img = img.astype(np.float32)

    # ---------- 1. 找亮像素 ----------
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    bright = (gray > highlight_thresh).astype(np.uint8)

    if not np.any(bright):
        return img.copy()

    # ---------- 2. Sobel 找细边 ----------
    sobelx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    sobely = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    grad = cv2.magnitude(sobelx, sobely)
    edge = (grad > grad_thresh).astype(np.uint8)

    # ---------- 3. 亮区边缘交集 ----------
    candidates = cv2.bitwise_and(bright, edge)

    # ---------- 4. 膨胀 + 羽化 ----------
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (edge_width, edge_width))
    band = cv2.dilate(candidates, kernel, iterations=1).astype(np.float32)
    band = cv2.GaussianBlur(band, (0, 0), sigmaX=edge_width * 0.6)

    # ---------- 5. 径向衰减 ----------
    y, x = np.ogrid[:h, :w]
    cx, cy = w / 2, h / 2
    dist = np.sqrt((x - cx) ** 2 + (y - cy) ** 2)
    radial = (dist / dist.max()) ** radial_gamma
    alpha = (band / band.max()) * radial * strength
    alpha = alpha[..., None]

    # ---------- 6. 叠加紫色 ----------
    purple = np.zeros_like(f_img)
    purple[..., 0], purple[..., 1], purple[..., 2] = 255, 100, 255  # BGR

    out = f_img * (1 - alpha) + purple * alpha
    return np.clip(out, 0, 255).astype(np.uint8)

In [ ]:
def auto_percentile_thresh(gray, pct=99.5):
    """返回灰度图 pct% 分位对应的阈值"""
    return np.percentile(gray, pct).astype(np.uint8)

bgr = cv2.imread("1.jpg")
out = add_subtle_purple_fringe(
    bgr,
    highlight_thresh=auto_percentile_thresh(cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY), 99.0),
    grad_thresh=25,
    edge_width=80,
    strength=0.7,
    radial_gamma=2.2
)
cv2.imwrite("flare.jpg", out)